# Same-genotype comparison demo

Compares multiple experiments that share a genotype: prints each experiment's genotype
up front (cross-checked against the h5, so a typo/stale label is caught immediately),
runs the same baseline QC as `2_data_quality_demo.ipynb` for each one individually, then
looks at how much they vary from each other -- cell-type yield, firing rates, DS/OS
cell counts, and contrast-response curves -- both per-experiment (to catch an outlier
day) and pooled (once you trust they agree).

Genotype isn't in the DataJoint schema yet -- `ra.get_genotype(exp_name)` reads it
straight from each h5's source properties. Most experiments don't have a genotype
recorded yet (see the genotype audit in `changes/`); this notebook only works for
experiments where it's been filled in.

In [ ]:
import retinanalysis as ra
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
%matplotlib inline

## Experiments to compare

Set `EXP_NAMES` to the experiments you want to pool, and `GENOTYPE` to what you expect
them all to be -- the next cell checks every experiment's actual h5 genotype against
this and flags any mismatch or missing value before you go any further.

In [ ]:
EXP_NAMES = [
    # e.g. '20260715A', '20260716A',
]
GENOTYPE = 'rd10'  # expected genotype shared across EXP_NAMES

In [ ]:
genotype_by_exp = {}
for exp_name in EXP_NAMES:
    found = ra.get_genotype(exp_name)
    genotype_by_exp[exp_name] = found
    flag = '' if found == GENOTYPE else '  <-- MISMATCH or missing'
    print(f'{exp_name}: {found!r}{flag}')

mismatches = [e for e, g in genotype_by_exp.items() if g != GENOTYPE]
if mismatches:
    print(f'\n{len(mismatches)} experiment(s) do not match GENOTYPE ({GENOTYPE!r}): {mismatches}')
    print('Fix EXP_NAMES/GENOTYPE or the h5 genotype field before trusting anything below.')
else:
    print(f'\nAll {len(EXP_NAMES)} experiments confirmed as {GENOTYPE!r}.')

## Baseline sanity checks (per experiment)

Same QC as `2_data_quality_demo.ipynb`, run once per experiment in `EXP_NAMES`: pick
each experiment's classified white-noise chunk, build a pipeline, and look at spike
counts and RF mosaics. Skim these before trusting the comparison figures below -- a bad
experiment pulled into a pooled/overlaid comparison is hard to spot after the fact.

In [ ]:
noise_search = ra.get_datasets_from_protocol_names('spatialnoise')

pipelines_by_exp = {}
for exp_name in EXP_NAMES:
    print(f'=== {exp_name} ===')
    datafile_name = ra.find_datafile_for_protocol(noise_search, exp_name)
    analysis_chunk_name = ra.find_classified_noise_chunk(exp_name)
    if datafile_name is None or analysis_chunk_name is None:
        print(f'{exp_name}: missing a datafile or classified chunk, skipping QC for this experiment.')
        continue
    with ra.scrollable_prints():
        pipeline = ra.create_mea_pipeline(exp_name, datafile_name, analysis_chunk_name=analysis_chunk_name)
    pipelines_by_exp[exp_name] = pipeline

In [ ]:
spike_xarr_by_exp = {}
for exp_name, pipeline in pipelines_by_exp.items():
    print(f'=== {exp_name}: spike counts ===')
    spike_times = ra.get_spike_xarr(pipeline.resp, minimum_n=3)
    spike_xarr_by_exp[exp_name] = spike_times
    spike_counts = xr.apply_ufunc(len, spike_times, vectorize=True)
    print(spike_counts.mean(dim=['epoch']).to_pandas())

In [ ]:
for exp_name, pipeline in pipelines_by_exp.items():
    pipeline.plot_rfs(minimum_n=3, b_zoom=True, title=f'{exp_name} ({GENOTYPE}) -- RFs by Cell Type')

## Cell-type yield across experiments

How many cells of each type got classified in each experiment -- catches a day where a
whole cell type just didn't show up, before that silently skews any pooled comparison.

In [ ]:
yield_rows = []
for exp_name, spike_times in spike_xarr_by_exp.items():
    counts = pd.Series(spike_times.coords['cell_type'].values).value_counts()
    for cell_type, n in counts.items():
        yield_rows.append({'exp_name': exp_name, 'cell_type': cell_type, 'n_cells': n})
df_yield = pd.DataFrame(yield_rows)

if len(df_yield) > 0:
    pivot_yield = df_yield.pivot(index='cell_type', columns='exp_name', values='n_cells').fillna(0)
    ax = pivot_yield.plot(kind='bar', figsize=(max(6, 0.6 * len(pivot_yield)), 4.5))
    ax.set_ylabel('N cells classified')
    ax.set_title(f'{GENOTYPE}: cell-type yield by experiment')
    ax.legend(title='exp_name', fontsize=8)
    plt.tight_layout()
    ra.scrollable_dataframe(pivot_yield)

## Firing-rate / responsiveness comparison

Mean spike count per cell type per experiment, during the white-noise recording used
for QC above -- flags a generally "quiet" recording relative to the others.

In [ ]:
rate_rows = []
for exp_name, spike_times in spike_xarr_by_exp.items():
    spike_counts = xr.apply_ufunc(len, spike_times, vectorize=True)
    mean_by_type = spike_counts.groupby(spike_times.coords['cell_type']).mean(...)
    for cell_type, val in zip(mean_by_type.coords['cell_type'].values, mean_by_type.values):
        rate_rows.append({'exp_name': exp_name, 'cell_type': cell_type, 'mean_spike_count': float(val)})
df_rates = pd.DataFrame(rate_rows)

if len(df_rates) > 0:
    pivot_rates = df_rates.pivot(index='cell_type', columns='exp_name', values='mean_spike_count').fillna(0)
    ax = pivot_rates.plot(kind='bar', figsize=(max(6, 0.6 * len(pivot_rates)), 4.5))
    ax.set_ylabel('Mean spike count per epoch')
    ax.set_title(f'{GENOTYPE}: responsiveness by experiment')
    ax.legend(title='exp_name', fontsize=8)
    plt.tight_layout()
    ra.scrollable_dataframe(pivot_rates)

## DS/OS cell counts across experiments

Reuses the exact DSI/OSI computation from `4_grating_dsos_demo.ipynb`
(`ra.compute_ds_os_summary`), run once per experiment, to compare how many
direction-/orientation-selective cells each experiment yielded.

In [ ]:
grating_search = ra.get_datasets_from_protocol_names('grating')
DS_OS_PROTOCOL_NAME = 'manookinlab.protocols.GratingDSOS'

In [ ]:
ds_os_summaries = {}
for exp_name in EXP_NAMES:
    summary = ra.compute_ds_os_summary(exp_name, DS_OS_PROTOCOL_NAME, grating_search)
    if summary is not None:
        ds_os_summaries[exp_name] = summary

df_ds_os_counts = pd.DataFrame([
    {'exp_name': s['exp_name'], 'n_cells': s['n_cells'], 'n_ds_cells': s['n_ds_cells'],
     'n_os_cells': s['n_os_cells'], 'pct_ds': s['pct_ds'], 'pct_os': s['pct_os']}
    for s in ds_os_summaries.values()
])
ra.scrollable_dataframe(df_ds_os_counts)

if len(df_ds_os_counts) > 0:
    ax = df_ds_os_counts.set_index('exp_name')[['pct_ds', 'pct_os']].plot(
        kind='bar', figsize=(max(6, 0.8 * len(df_ds_os_counts)), 4.5)
    )
    ax.set_ylabel('% of cells')
    ax.set_title(f'{GENOTYPE}: DS/OS cell yield by experiment')
    plt.tight_layout()

In [ ]:
df_metrics_all = pd.concat([s['df_metrics'] for s in ds_os_summaries.values()], ignore_index=True) \
    if ds_os_summaries else pd.DataFrame(columns=['exp_name', 'dsi', 'osi'])

if len(df_metrics_all) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    df_metrics_all.boxplot(column='dsi', by='exp_name', ax=axes[0])
    axes[0].set_title('DSI by experiment'); axes[0].set_xlabel(''); axes[0].set_ylabel('DSI')
    df_metrics_all.boxplot(column='osi', by='exp_name', ax=axes[1])
    axes[1].set_title('OSI by experiment'); axes[1].set_xlabel(''); axes[1].set_ylabel('OSI')
    fig.suptitle(f'{GENOTYPE}: DSI/OSI distribution across experiments')
    plt.tight_layout()

## Contrast comparison

First, per-experiment CRF curves overlaid on the same axes (`ra.plot_crf_overlay_by_experiment`)
-- this is the direct check for whether experiments actually agree before pooling them.
Then the pooled comparison (`ra.plot_crf_across_experiments`) and a c50 distribution
across experiments, once the overlay above looks reasonable.

In [ ]:
contrast_search = ra.get_datasets_from_protocol_names('contrast')
GRATING_PROTOCOL_NAME = 'manookinlab.protocols.ContrastResponseGrating'
CORR_THRESHOLD = 0.8

In [ ]:
figs_overlay = ra.plot_crf_overlay_by_experiment(
    EXP_NAMES, contrast_search, GRATING_PROTOCOL_NAME, condition_key='contrast',
    corr_cutoff=CORR_THRESHOLD, response_col='f1',
    baseline_condition_key='contrast', baseline_condition_value=0.0,
    title_prefix=f'{GENOTYPE}: per-experiment CRF overlay', even_spacing=True,
)

In [ ]:
figs_crf, df_crf_points = ra.plot_crf_across_experiments(
    EXP_NAMES, contrast_search, GRATING_PROTOCOL_NAME, condition_key='contrast',
    corr_cutoff=CORR_THRESHOLD, response_col='f1',
    baseline_condition_key='contrast', baseline_condition_value=0.0,
    title_prefix=f'{GENOTYPE}: F1 vs. contrast across NDFs, pooled across experiments',
    even_spacing=True,
)
ra.scrollable_dataframe(df_crf_points)

In [ ]:
df_c50_fits = ra.get_naka_rushton_fits_across_experiments(
    EXP_NAMES, contrast_search, GRATING_PROTOCOL_NAME, corr_cutoff=CORR_THRESHOLD,
)
df_c50_well_constrained = df_c50_fits[df_c50_fits['well_constrained']] if len(df_c50_fits) else df_c50_fits

if len(df_c50_well_constrained) > 0:
    fig, ax = plt.subplots(figsize=(max(5, 0.8 * df_c50_well_constrained['exp_name'].nunique()), 4.5))
    df_c50_well_constrained.boxplot(column='c50', by='exp_name', ax=ax)
    ax.set_title(f'{GENOTYPE}: c50 distribution across experiments (well-constrained fits only)')
    ax.set_xlabel(''); ax.set_ylabel('c50')
    plt.suptitle('')
    plt.tight_layout()
    ra.scrollable_dataframe(df_c50_well_constrained)

## Summary table

One row per experiment: cell yield, DS/OS counts, and median c50 -- an at-a-glance
"did this day look normal" table.

In [ ]:
summary_rows = []
for exp_name in EXP_NAMES:
    n_cells_total = int(df_yield.query('exp_name == @exp_name')['n_cells'].sum()) if len(df_yield) else None
    ds_os_row = df_ds_os_counts.query('exp_name == @exp_name') if len(df_ds_os_counts) else pd.DataFrame()
    c50_vals = df_c50_well_constrained.query('exp_name == @exp_name')['c50'] if len(df_c50_well_constrained) else pd.Series(dtype=float)
    summary_rows.append({
        'exp_name': exp_name,
        'genotype': genotype_by_exp.get(exp_name),
        'n_cells_classified': n_cells_total,
        'n_ds_cells': int(ds_os_row['n_ds_cells'].iloc[0]) if len(ds_os_row) else None,
        'n_os_cells': int(ds_os_row['n_os_cells'].iloc[0]) if len(ds_os_row) else None,
        'median_c50': float(c50_vals.median()) if len(c50_vals) else None,
        'n_c50_fits': len(c50_vals),
    })
df_summary = pd.DataFrame(summary_rows)
ra.scrollable_dataframe(df_summary)